# Model Evaluation

## Agenda
- Cross validation
- Regularization
- Hyperparameter search
- More classification metrics
- Missing value imputation


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import urllib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


## Preparing the data


In [2]:
url = 'https://pub-c88b3a7f2ed141418355b2bfb03c96e6.r2.dev/income.csv'
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response:
    df = pd.read_csv(response)


In [3]:
df.sample(10)

,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
37097,41,Self-emp-not-inc,12th,8.0,Divorced,Craft-repair,Unmarried,White,1,0.0,0.0,40.0,United-States,0
38354,23,State-gov,Assoc-voc,11.0,Married-AF-spouse,Adm-clerical,Wife,White,0,0.0,0.0,30.0,United-States,0
28324,32,Private,Bachelors,13.0,Never-married,Prof-specialty,Not-in-family,White,1,0.0,0.0,38.0,United-States,0
10953,31,Self-emp-not-inc,Bachelors,13.0,Divorced,Exec-managerial,Unmarried,White,1,0.0,0.0,60.0,United-States,1
39382,44,Self-emp-not-inc,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,1,0.0,0.0,35.0,Germany,0
43601,20,Private,Some-college,10.0,Never-married,Other-service,Own-child,White,1,0.0,0.0,30.0,United-States,0
27548,19,Private,12th,8.0,Never-married,Handlers-cleaners,Not-in-family,White,1,0.0,0.0,40.0,United-States,0
35997,23,Private,HS-grad,9.0,Never-married,Sales,Own-child,Black,1,0.0,0.0,20.0,United-States,0
32276,29,Local-gov,Some-college,10.0,Never-married,Protective-serv,Own-child,White,1,0.0,0.0,48.0,United-States,0
7913,29,Local-gov,HS-grad,9.0,Separated,Protective-serv,Other-relative,White,0,0.0,0.0,40.0,United-States,0


In [4]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
df_train, df_test = train_test_split(df, test_size=0.2, random_state=10)

## Missing values

Most algorithms don't work when missing values are present in the data. And in our dataset we have a few rows that contain missing values in some of the columns.

In [5]:
# Count rows with at least one missing value
df_train.isna().any(axis=1).sum()

np.int64(2909)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             48842 non-null  int64  
 1   workclass       46043 non-null  object 
 2   education       48842 non-null  object 
 3   education_num   48842 non-null  float64
 4   marital_status  48842 non-null  object 
 5   occupation      46033 non-null  object 
 6   relationship    48842 non-null  object 
 7   race            48842 non-null  object 
 8   sex             48842 non-null  int64  
 9   capital_gain    48842 non-null  float64
 10  capital_loss    48842 non-null  float64
 11  hours_per_week  48842 non-null  float64
 12  native_country  47985 non-null  object 
 13  income          48842 non-null  int64  
dtypes: float64(4), int64(3), object(7)
memory usage: 5.2+ MB


We don't just want to remove those rows, they're still useful! Also, missing data does not necessarily imply that there's something wrong with the row. 

Imputing missing values for categorical features is relatively simple - just treat the missing values as a separate category.

In [7]:
df_train.value_counts("workclass", dropna=False)

workclass
 Private             27235
 Self-emp-not-inc     3081
 Local-gov            2487
NaN                   2255
 State-gov            1550
 Self-emp-inc         1310
 Federal-gov          1133
 Without-pay            14
 Never-worked            8
Name: count, dtype: int64

In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Now we can use the imputer if needed for other missing values
missing_imputer = SimpleImputer(strategy="constant", fill_value="missing")
workclass_imputed = missing_imputer.fit_transform(df_train[["workclass"]])

# One-hot encode the imputed data
encoder = OneHotEncoder(handle_unknown="ignore")
workclass_encoded = encoder.fit_transform(workclass_imputed)

# continue with modelling

How can we impute continuous columns? All numerical columns in the dataset we're currently using have no missing data, so I'll create a fake feature with some data from a uniform distribution and some missing values.

In [9]:
feature = np.random.uniform(size=50)
feature[:10] = np.nan

tmp = pd.DataFrame(feature, columns=["a"])

In [10]:
mean_imputer = SimpleImputer(strategy="mean")
mean_imputer.fit_transform(tmp)[:20]

array([[0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.53984679],
       [0.55748683],
       [0.28385307],
       [0.315662  ],
       [0.67988434],
       [0.6206953 ],
       [0.17389806],
       [0.8874693 ],
       [0.13844005],
       [0.55404956],
       [0.92851593]])

The feature was imputed with the mean of values in rows where data is not missing. "median" is another strategy we could have used here.

### Task

Fit a LogisticRegression model for predicting `income` using these features:
- `workclass`
- `hours_per_week`
- `relationship`
- `age`

For categorical variables, apply missing value imputation (strategy `constant`) and one-hot encoding. To continuous variables, apply PolynomialFeatures(degree=2) and StandardScaler.

Set penalty in LogisticRegression to "l1".

Use ColumnTransformer from last week to select which features to apply the transformations to.


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression

categorical_features = ["relationship", "workclass"]
numerical_features = ["hours_per_week",  "age"]

X_train = df_train[categorical_features + numerical_features]
y_train = df_train["income"]

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

numerical_transformer = Pipeline(
    steps=[("poly", PolynomialFeatures(degree=2)), ("scaler", StandardScaler())]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ]
)

model = Pipeline(
    steps=[("preprocessor", preprocessor), ("logreg", LogisticRegression(penalty="l1", solver="liblinear"))]
)

model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

## Putting it all together

Now we have all the tools that we need in order to successfully fit a classification or regression model.
- We know some algorithms (logistic regression, K-Nearest-Neighbor).
- We know some useful feature preprocessing steps:
  - One-hot encoding, that allows us to use categorical features in our models.
  - Polynomial transformations, that are useful for increasing the flexibility of the logistic regression model.
  - Feature scaling (minmax scaling and standard scaling), that improve the performance of most algorithms.
  - Missing value imputation.
- We know what regularization is and how it helps prevent overfitting.
- We know several metrics that allow us to measure how good our model performs.
- We know how to perform cross validation to validate how any specific model performs (in terms of the metric we chose), so as to choose the best model.
- We know what hyperparameters are and how to tune them with the help of cross validation.
- We know that we should keep a portion of our data for reporting the performance of our final model.

We can put all of these steps together to fit a very useful model. Let's do that step by step.


### Step 1

Fit a LogisticRegression model with penalty=None that predicts `income` from `hours_per_week` and `education_num`. Evaluate the model via cross-validation using roc_auc as the metric.

Note that we're beginning with something very basic - a simple algorithm with just two features that have no missing values.


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

numerical_features = ["hours_per_week", "education_num"]

X_train = df_train[numerical_features]
y_train = df_train["income"]

model = LogisticRegression(penalty=None)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
np.mean(scores)

np.float64(0.7594505621289634)

### Step 2

Modify the code above by adding the `occupation` variable among the predictors. Keep in mind that it's a categorical variable with missing values. Use scikit-learn Pipeline and ColumnTransformer.


In [13]:
# Hint

# import

# select features

categorical_transformer = Pipeline(
    steps=[
        ...
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, ["feat1", "feat2"]),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", ...),
    ]
)

# rest is the same

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numerical_features = ["hours_per_week", "education_num"]
categorical_features = ["occupation"]

X_train = df_train[numerical_features + categorical_features]
y_train = df_train["income"]

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(drop="first")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty=None, max_iter=5000)),
    ]
)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
np.mean(scores)

np.float64(0.7860601672208242)

### Step 3

Modify the code above to apply PolynomialFeatures(degree=2) and StandardScaler to `hours_per_week` and `education_num` features.


In [15]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# same features as before

numerical_transformer = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2)),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ],
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty=None, max_iter=5000)),
    ]
)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
np.mean(scores)

np.float64(0.7884730628425214)

### Step 4

Modify the code above by adding regularization to the model. Use l1 regularization, do not set a value for C. Remember to change the solver to "liblinear".


In [16]:

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty="l1", max_iter=5000, solver="liblinear")),
    ]
)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
np.mean(scores)

np.float64(0.7884767200717437)

### Step 5

Modify the code above to find an optimal value for C by performing hyperparameter tuning. Use `GridSearchCV`.


In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {"logreg__C": [10**i for i in range(-3, 3)]}
grid_search = GridSearchCV(...)
grid_search.fit(X_train, y_train)

print(...)


TypeError: GridSearchCV.__init__() missing 1 required positional argument: 'param_grid'

In [ ]:
from sklearn.model_selection import GridSearchCV

# same features as before

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(drop="first")),
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2)),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ],
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty="l1", max_iter=5000, solver="liblinear")),
    ]
)

param_grid = {"logreg__C": [10**i for i in range(-3, 3)]}

grid_search = GridSearchCV(model, param_grid, cv=5, scoring="roc_auc")
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(round(grid_search.best_score_, 4))

{'logreg__C': 10}
0.7885


### Step 6

Add `native_country` feature to the model. Notice that this column has 41 categories, and that it contains missing values - apply missing value imputation and use max_categories=10 in OneHotEncoder.


In [19]:
df_train["native_country"].nunique()

41

In [20]:
df_train.value_counts("native_country", dropna=False)

native_country
 United-States                 35049
 Mexico                          768
NaN                              684
 Philippines                     236
 Germany                         167
 Canada                          144
 Puerto-Rico                     143
 El-Salvador                     121
 Cuba                            114
 India                           108
 England                         107
 China                            94
 Dominican-Republic               89
 South                            89
 Jamaica                          87
 Italy                            83
 Columbia                         75
 Vietnam                          73
 Japan                            71
 Guatemala                        71
 Poland                           68
 Haiti                            64
 Portugal                         56
 Iran                             48
 Taiwan                           47
 Nicaragua                        43
 Greece                

In [21]:
numerical_features = ["hours_per_week", "education_num"]
categorical_features = ["occupation", "native_country"]

X_train = df_train[numerical_features + categorical_features]

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(drop="first", max_categories=10, handle_unknown="ignore")),
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2)),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty="l1", max_iter=5000, solver="liblinear")),
    ]
)



In [22]:
param_grid = {"logreg__C": [10**i for i in range(-6, 5)]}

grid_search = GridSearchCV(model, param_grid, cv=5, scoring="roc_auc")
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(round(grid_search.best_score_, 4))


{'logreg__C': 10000}
0.7855


scikit-learn is warning me about unknown categories in the data. This happens because during cross validation our validation set sometimes contains countries that are not in the training set. We can ignore this warning, since we set `handle_unknown="ignore"` in OneHotEncoder, which handles such cases.

I also elected to limit the number of categories to 10 biggest categories. OneHotEncoder puts the rest of the values in some "other" category.


### Step 7

Add these features to the model:
- `age`
- `race`

Apply the same transformations as we did so far for categorical and numerical features.

In [23]:
categorical_features = ["race", "occupation", "native_country"]
numerical_features = ["hours_per_week", "education_num", "age"]

X_train = df_train[numerical_features + categorical_features]
y_train = df_train["income"]


In [24]:

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(drop="first", max_categories=10, handle_unknown="ignore")),
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2)),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numerical_transformer, numerical_features),
    ])

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty="l1", max_iter=5000, solver="liblinear")),
    ]
)
model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

### Step 8

Find optimal values for C and class_weight between `None` and `"balanced"`.


In [25]:
param_grid = {"logreg__C": [10**i for i in range(-4, 4)], "logreg__class_weight": [None, "balanced"]}

grid_search = GridSearchCV(model, param_grid, cv=5, scoring="roc_auc")
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(round(grid_search.best_score_, 4))


{'logreg__C': 100, 'logreg__class_weight': None}
0.8301


### Step 9

Train a model with the best hyperparameters from the last step.

Report the performance of the model on the test dataset.




In [26]:
X_test = df_test[numerical_features + categorical_features]
y_test = df_test["income"]

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(penalty="l1", C=100, class_weight=None, max_iter=5000, solver="liblinear")),
    ]
)
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [27]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

y_score = model.predict_proba(X_test)[:, 1]

print(roc_auc_score(y_test, y_score))

y_pred = y_score > 0.5
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))
print(f1_score(y_test, y_pred))


0.8264293472726078
0.6538205980066445
0.4124056999161777
0.5057825751734772


And that is what modelling looks like! We began with almost the simplest model possible and gradually added complexity while also making the model better. Then we reported the model's performance on test data.
